# Nanobody binder design with BoltzGen
*Designing de novo nanobody binders to PDL1 using BoltzGen and proteinMPNN*

In this tutorial, we'll demonstrate how to use the OpenProtein.AI Python client to
design a nanobody that binds to a target protein. We refer to the designed protein as the
**binder** and the protein being bound as the **target**.

Unlike general protein binder design, for nanobody design we utilize a **scaffold-based approach**. We will start with an existing nanobody framework and essentially "graft" new Complementarity-Determining Regions (CDRs) onto it. This ensures that our designed binder retains the stable, expressible framework regions of a natural nanobody while tailoring the binding loops (CDRs) to our specific target. The design process consists of four main steps:

1. **Query Specification**: Specify the design problem as a "query", including
    1. the target protein (PDL1)
    2. the nanobody scaffold (framework regions)
    3. the lengths of the CDR loops to be designed

2. **Structure Generation**: Generate plausible structures for the nanobody binder CDRs using
   **BoltzGen** (([Stark et al., 2025](https://www.biorxiv.org/content/10.1101/2025.11.20.689494v1))), a generative model capable of designing backbone structures using scaffolds.

3. **Sequence Design**: Design sequences for the generated CDRs using **proteinMPNN** ([Dauparas et al., 2022](https://doi.org/10.1126/science.add2187)),
   an inverse folding model for generating the binder sequence conditioned on the generated structure.

4. **In Silico Validation**: Validate the designs by predicting their structures with **Boltz-2** ([Passaro et al., 2025](https://www.biorxiv.org/content/10.1101/2025.06.14.659707v1))
   and computing metrics to select the best candidates for experimental testing.

# Prerequisites

To run this tutorial, you'll need a Python environment containing the following
packages:

- `openprotein_python>=0.10`
- `molviewspec` (for structure visualization)

See the Python client [installation instructions](https://docs.openprotein.ai/python-api/installation.html) for more info.

Additionally, you should have your [credentials set up](https:/docs.openprotein.ai/python-api/quickstart.html) in `~/.openprotein/config.toml` to
authenticate with the OpenProtein.AI API.

## Import necessary packages

In [5]:
import io
import requests
from dataclasses import dataclass

import numpy as np
import numpy.typing as npt
import pandas as pd
from scipy.spatial.transform import Rotation

from tqdm import tqdm

import molviewspec as mvs
from molviewspec.nodes import RepresentationTypeT

import openprotein
from openprotein.fasta import parse_stream
from openprotein.molecules import Protein, Complex, Structure

/Users/tbepler/miniconda3/envs/openprotein/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_index" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/Users/tbepler/miniconda3/envs/openprotein/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_id" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


## Connect to OpenProtein.AI

In [6]:
session = openprotein.connect()
print("✅ Successfully connected to the OpenProtein.AI API!")

✅ Successfully connected to the OpenProtein.AI API!


# Step 1: Binder Design Problem Specification
*Specify the nanobody binder design problem*

In this tutorial, we will design nanobody binders against **Programmed Death-Ligand 1 (PDL1)**. This design problem is adapted from the BoltzGen study ([Stark et al., 2025](https://www.biorxiv.org/content/10.1101/2025.11.20.689494v1)).

We will design a **nanobody binder**, which is a single-domain antibody fragment derived from heavy-chain-only antibodies found in camelids. To restrict the BotlzGen structure generator to specifically design nanobody binders, we will use a **scaffold**. The scaffold defines an overall framework structure and specific designable regions for binding to the target. This means we will keep the framework regions of an existing, well-behaved nanobody constant, while redesigning the Complementarity Determining Regions (CDRs) to bind our specific target. Later, we will use proteinMPNN to fill in the CDR sequences conditioned on the generated binder CDR structures.

The **scaffold** provides convenient ways to generate binders of other types such as scFvs for FAbs.

_Note_: we are using proteinMPNN for inverse folding the CDRs here, but we could use other generative models such as **PoET-2** instead. PoET-2 is unique in its ability to use **prompt context** sequences that define specific families of proteins, e.g., human VHH domains, that can be used to guide the generated proteins towards specific characteristics. This is especially useful if we want to redesign the whole binder sequence. Because we are only redesigning the CDRs here, we'll use proteinMPNN for simplicity. [Learn more about PoET-2](https://www.openprotein.ai/a-multimodal-foundation-model-for-controllable-protein-generation-and-representation-learning).

## Step 1.1: Define and visualize the target

We fetch the structure of the PDL1 target from PDB.

In [9]:
structure = Structure.from_pdb_id('7uxq') # PDL1
first_complex = structure[0]
target = first_complex.get_protein(chain_id="A")
print(target.formatted(include=("sequence", "structure_mask")))

Unknown amino acid at position 1: ACE
Residue at position 1 missing backbone atom=N
Residue at position 1 missing backbone atom=CA
Unknown amino acid at position 1: ACE
Residue at position 1 missing backbone atom=N
Residue at position 1 missing backbone atom=CA
0     SEQUENCE       MAFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNIIQFVHGEEDLKV
0     STRUCTURE_MASK ^                                                           

60    SEQUENCE       QHSSYRQRARLLKDQLSLGNAALQITDVKLQDAGVYRCMISYGGADYKRITVKVNAPYAA
60    STRUCTURE_MASK                                                             

120   SEQUENCE       ALEHHHHHH
120   STRUCTURE_MASK         ^


Remove the His tag and linker since we don't want to bind that.

In [10]:
target = target[:len(target) - 8]
print(target.formatted(include=("sequence", "structure_mask")))

0     SEQUENCE       MAFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNIIQFVHGEEDLKV
0     STRUCTURE_MASK ^                                                           

60    SEQUENCE       QHSSYRQRARLLKDQLSLGNAALQITDVKLQDAGVYRCMISYGGADYKRITVKVNAPYAA
60    STRUCTURE_MASK                                                             

120   SEQUENCE       A
120   STRUCTURE_MASK  


## Step 1.2: Define the nanobody scaffold

We will use the structure from PDB ID `7eow` as our scaffold. Thie scaffold is from *caplacizumab*, a humanized VHH. We'll use the structure and framework region of this VHH as the scaffold for our binder, but design new CDRs for binding to our target.

First, we load the protein.

In [11]:
structure = Structure.from_pdb_id("7eow")
first_complex = structure[0]
binder_scaffold = first_complex.get_protein(chain_id="B")
print(binder_scaffold.formatted(include=("sequence", "structure_mask")))

0     SEQUENCE       MEVQLVESGGGLVQPGGSLRLSCAASGRTFSYNPMGWFRQAPGKGRELVAAISRTGGSTY
0     STRUCTURE_MASK ^                                                           

60    SEQUENCE       YPDSVEGRFTISRDNAKRMVYLQMNSLRAEDTAVYYCAAAGVRAEDGRVRTLPSEYTFWG
60    STRUCTURE_MASK                                                             

120   SEQUENCE       QGTQVTVSSLEHHHHHH
120   STRUCTURE_MASK          ^^^^^^^^


### Clean the scaffold

We remove the leading Methionine (M) and the trailing Histidine tag (His-tag) because they are expression artifacts. The structure mask confirms these residues have no defined structure.

In [12]:
binder_scaffold = binder_scaffold[~binder_scaffold.get_structure_mask()]
print(binder_scaffold.formatted(include=("sequence", "structure_mask")))

0     SEQUENCE       EVQLVESGGGLVQPGGSLRLSCAASGRTFSYNPMGWFRQAPGKGRELVAAISRTGGSTYY
0     STRUCTURE_MASK                                                             

60    SEQUENCE       PDSVEGRFTISRDNAKRMVYLQMNSLRAEDTAVYYCAAAGVRAEDGRVRTLPSEYTFWGQ
60    STRUCTURE_MASK                                                             

120   SEQUENCE       GTQVTVSS
120   STRUCTURE_MASK         


### Define the framework and binding regions

We want to use the nanobody structure as a framework, but design new CDRs for binding to our target. To do this, we keep the framework regions (FWRs) constant but replace the CDRs with designable regions.

In this example, we will set CDR1 length to 10 (increased from 9), CDR2 to 8 (same as scaffold), and CDR3 to 20 (decreased from 21). The `X` characters represent residues to be designed.

In [13]:
fwr1 = binder_scaffold[:25]
fwr2 = binder_scaffold[34:51]
fwr3 = binder_scaffold[59:97]
fwr4 = binder_scaffold[118:]
print("FWR1:", fwr1.sequence.decode())
print("FWR2:", fwr2.sequence.decode())
print("FWR3:", fwr3.sequence.decode())
print("FWR4:", fwr4.sequence.decode())

FWR1: EVQLVESGGGLVQPGGSLRLSCAAS
FWR2: GWFRQAPGKGRELVAAI
FWR3: YPDSVEGRFTISRDNAKRMVYLQMNSLRAEDTAVYYCA
FWR4: GQGTQVTVSS


In [14]:
cdr1_length = 10
cdr2_length = 8
cdr3_length = 20
binder_scaffold = (
    fwr1
    + "X" * cdr1_length
    + fwr2
    + "X" * cdr2_length
    + fwr3
    + "X" * cdr3_length
    + fwr4
)
print(binder_scaffold.formatted(include=("sequence", "structure_mask")))

0     SEQUENCE       EVQLVESGGGLVQPGGSLRLSCAASXXXXXXXXXXGWFRQAPGKGRELVAAIXXXXXXXX
0     STRUCTURE_MASK                          ^^^^^^^^^^                 ^^^^^^^^

60    SEQUENCE       YPDSVEGRFTISRDNAKRMVYLQMNSLRAEDTAVYYCAXXXXXXXXXXXXXXXXXXXXGQ
60    STRUCTURE_MASK                                       ^^^^^^^^^^^^^^^^^^^^  

120   SEQUENCE       GTQVTVSS
120   STRUCTURE_MASK         


## Step 1.3: Configure relative positioning (Groups)

By default, all residues are in "group 0", which implies their relative positions are fixed. Since we want the nanobody to dock against the target (i.e., its position relative to the target is not fixed), we assign the scaffold to a different group (group 1).

In [15]:
# Visualize current groups (all 0)
print("\nVisualize target groups:")
print(target.formatted(("sequence", "group")))
print("\nVisualize binder scaffold groups:")
print(binder_scaffold.formatted(("sequence", "group")))


Visualize target groups:
0     SEQUENCE MAFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNIIQFVHGEEDLKV
0     GROUP    000000000000000000000000000000000000000000000000000000000000

60    SEQUENCE QHSSYRQRARLLKDQLSLGNAALQITDVKLQDAGVYRCMISYGGADYKRITVKVNAPYAA
60    GROUP    000000000000000000000000000000000000000000000000000000000000

120   SEQUENCE A
120   GROUP    0

Visualize binder scaffold groups:
0     SEQUENCE EVQLVESGGGLVQPGGSLRLSCAASXXXXXXXXXXGWFRQAPGKGRELVAAIXXXXXXXX
0     GROUP    000000000000000000000000000000000000000000000000000000000000

60    SEQUENCE YPDSVEGRFTISRDNAKRMVYLQMNSLRAEDTAVYYCAXXXXXXXXXXXXXXXXXXXXGQ
60    GROUP    000000000000000000000000000000000000000000000000000000000000

120   SEQUENCE GTQVTVSS
120   GROUP    00000000


In [16]:
# Set scaffold to group 1 to unfix relative position
binder_scaffold = binder_scaffold.set_group(1)
print("\nUpdated binder scaffold groups:")
print(binder_scaffold.formatted(("sequence", "group")))


Updated binder scaffold groups:
0     SEQUENCE EVQLVESGGGLVQPGGSLRLSCAASXXXXXXXXXXGWFRQAPGKGRELVAAIXXXXXXXX
0     GROUP    111111111111111111111111111111111111111111111111111111111111

60    SEQUENCE YPDSVEGRFTISRDNAKRMVYLQMNSLRAEDTAVYYCAXXXXXXXXXXXXXXXXXXXXGQ
60    GROUP    111111111111111111111111111111111111111111111111111111111111

120   SEQUENCE GTQVTVSS
120   GROUP    11111111


Finally, we combine the target and the binder scaffold into a single `Complex` query.

In [17]:
query = target & binder_scaffold
print("Query type", type(query))
print("Chains in query:", list(query.get_chains().keys()))
print("\nVisualize target (Chain A):")
print(query.get_protein(chain_id="A").formatted(include=("sequence", "structure_mask")))
print("\nVisualize binder scaffold (Chain B):")
print(query.get_protein(chain_id="B").formatted(include=("sequence", "structure_mask")))

Query type <class 'openprotein.molecules.complex.Complex'>
Chains in query: ['A', 'B']

Visualize target (Chain A):
0     SEQUENCE       MAFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNIIQFVHGEEDLKV
0     STRUCTURE_MASK ^                                                           

60    SEQUENCE       QHSSYRQRARLLKDQLSLGNAALQITDVKLQDAGVYRCMISYGGADYKRITVKVNAPYAA
60    STRUCTURE_MASK                                                             

120   SEQUENCE       A
120   STRUCTURE_MASK  

Visualize binder scaffold (Chain B):
0     SEQUENCE       EVQLVESGGGLVQPGGSLRLSCAASXXXXXXXXXXGWFRQAPGKGRELVAAIXXXXXXXX
0     STRUCTURE_MASK                          ^^^^^^^^^^                 ^^^^^^^^

60    SEQUENCE       YPDSVEGRFTISRDNAKRMVYLQMNSLRAEDTAVYYCAXXXXXXXXXXXXXXXXXXXXGQ
60    STRUCTURE_MASK                                       ^^^^^^^^^^^^^^^^^^^^  

120   SEQUENCE       GTQVTVSS
120   STRUCTURE_MASK         


## Visualizing structure of query...

In [18]:
@dataclass(frozen=True)
class ColorSpec:
    chain_id: str
    color: str
    positions: list[int] | None = None
    rep_type: RepresentationTypeT = "cartoon"


def visualize_cif(cif_string: str, colors: list[ColorSpec]):
    builder = mvs.create_builder()
    model = (
        builder.download(url="structure.cif").parse(format="mmcif").model_structure()
    )
    for color_spec in colors:
        component = model.component(
            selector=(
                mvs.ComponentExpression(label_asym_id=color_spec.chain_id)
                if color_spec.positions is None
                else [
                    mvs.ComponentExpression(
                        label_asym_id=color_spec.chain_id, label_seq_id=i
                    )
                    for i in color_spec.positions
                ]
            )
        )
        rep = component.representation(type=color_spec.rep_type)
        rep.color(color=color_spec.color)
    builder.molstar_notebook(
        data={"structure.cif": cif_string},
        width=600,
        height=500,
    )

In [71]:
visualize_cif(
    cif_string=query.to_string(),
    colors=[
        # color the target chain a light blue
        ColorSpec(chain_id="A", color="#b5e2f5"),  # target in blue
        ColorSpec(chain_id="B", color="#f4c30b"),  # binder scaffold in orange
    ],
)

<IPython.core.display.Javascript object>

# Step 2: Structure Generation with BoltzGen
*Generate plausible structures for the nanobody binder*

We use **BoltzGen**, a structure generation model that supports scaffolds, to generate plausible backbone structures for our nanobody that complement the target.

In [72]:
N_STRUCTURES = 100

# 1. Create the BoltzGen job
boltzgen_job = session.models.boltzgen.generate(query=query, N=N_STRUCTURES)
print(boltzgen_job)

# 2. Wait for the job to finish
_ = boltzgen_job.wait_until_done(verbose=True)

# 3. Get results
generated_structures: list[Complex] = boltzgen_job.get()
print(f"Generated {len(generated_structures)} structures.")

job_id='06246f4b-01b7-41b0-b874-c455e8a1aea7' job_type='/models/boltzgen' status=<JobStatus.PENDING: 'PENDING'> created_date=datetime.datetime(2026, 1, 19, 17, 47, 44, 361815, tzinfo=TzInfo(UTC)) start_date=None end_date=None prerequisite_job_id=None progress_message=None progress_counter=0 sequence_length=None


Waiting: 100%|██████████| 100/100 [33:48<00:00, 20.28s/it, status=SUCCESS] 


Generated 100 structures.


# Step 3: Sequence Design with proteinMPNN
*Design the CDR sequences using inverse folding*

BoltzGen generates the backbone structure and an initial sequence, but we can often improve the sequence quality (e.g., expression, stability) using an inverse folding model.

We will use **proteinMPNN**, an inverse folding model commonly used for generating amino acid sequences likely to fold into a defined backbone structure.

## 3.1 Generate sequences

We now iterate through our generated structures and redesign the CDR regions. In this example, we are only infilling the sequence of the CDRs while keeping the sequence of the framework region from the scaffold. We utilize the `binder_scaffold` sequence to mask the CDR regions (where the scaffold has `X`), ensuring we only redesign those specific areas while preserving the backbone structure.

We could mask the sequence of the complete nanobody to generate the sequence for both the framework and CRDs, but proteinMPNN will not generate natural-like framework regions. For this, we recommend using PoET-2 with a human or camelid VHH context.

**Note:** To inverse fold multichain complexes, we must join the target and binder with a linker (e.g., `GGGGS*3`).

In [ ]:
N_SEQS_PER_STRUCTURE = 10

# 1. Create the jobs
gen_jobs = []
for generated_structure in tqdm(
    generated_structures, mininterval=1.0, desc="Creating jobs"
):
    # 1a. Create protein for inverse folding by joining the generated target and binder
    #     with a linker
    target_for_inverse_folding = generated_structure.get_protein(chain_id="A")
    binder_for_inverse_folding = (
        generated_structure.get_protein(chain_id="B")
        .copy()
        .set_sequence(binder_scaffold.sequence) # use the scaffold framework sequence
        .mask_structure(side_chain_only=True) # mask the side chain atoms since we want to generate these sequences
    )
    linker = "GGGGS" * 3
    protein_for_inverse_folding = (
        target_for_inverse_folding + linker + binder_for_inverse_folding
    )
    # 1b. Create the sequence generation job and store the job
    gen_job = session.models.proteinmpnn.generate(
        query=protein_for_inverse_folding,
        num_samples=N_SEQS_PER_STRUCTURE,
        # use a low temperature for higher quality samples at the cost of lower diversity
        temperature=0.1,
        seed=42,  # for reproducibility
    )
    gen_jobs.append(gen_job)

# 2. Wait for the jobs to finish
for gen_job in tqdm(gen_jobs, mininterval=1.0, desc="Waiting for jobs"):
    _ = gen_job.wait_until_done()
    assert gen_job.status == "SUCCESS"

Waiting for jobs: 100%|██████████| 100/100 [06:45<00:00,  4.05s/it]


In [ ]:
# Collect all designed sequences into a DataFrame
records = []
for i, gen_job in enumerate(tqdm(gen_jobs, mininterval=1.0)):
    for j, poet_result in enumerate(gen_job.get()):
        # The result from PoET-2 includes target+linker+binder.
        # We need to extract just the binder sequence.
        full_sequence = poet_result.sequence

        # The binder is at the end.
        # Length of target = len(target)
        # Length of linker = 15 ("GGGGS" * 3)
        binder_start_index = len(target) + 15
        designed_binder_sequence = full_sequence[binder_start_index:]

        records.append(
            {
                "design_idx": i * N_SEQS_PER_STRUCTURE + j,
                "structure_idx": i,
                "sequence_idx": j,
                "score": poet_result.score.mean().item(),
                "sequence": designed_binder_sequence,
            }
        )
df = pd.DataFrame.from_records(records).set_index(["structure_idx", "sequence_idx"])
df.head()

100%|██████████| 100/100 [00:36<00:00,  2.73it/s]


design_idx       score  \
structure_idx sequence_idx                           
0             0                      0 -113.704020   
              1                      1 -115.618840   
              2                      2 -112.545610   
              3                      3 -120.459770   
              4                      4 -117.207985   

                                                                     sequence  
structure_idx sequence_idx                                                     
0             0             EVQLVESGGGLVQPGGSLRLSCAASGNFDFFAKLYGWFRQAPGKGR...  
              1             EVQLVESGGGLVQPGGSLRLSCAASGNFDFFAKYYGWFRQAPGKGR...  
              2             EVQLVESGGGLVQPGGSLRLSCAASGNYDFFSKLYGWFRQAPGKGR...  
              3             EVQLVESGGGLVQPGGSLRLSCAASGNFDFFAKLAGWFRQAPGKGR...  
              4             EVQLVESGGGLVQPGGSLRLSCAASGNFDFFARYYGWFRQAPGKGR...

# Step 4: In Silico Validation
*Validate designs using structure prediction*

Finally, we validate our designs by predicting the structure of the designed sequences using **Boltz-2**. We will predict the complex structure and compute metrics to filter for high-confidence designs.

## 4.1 Predict structures with Boltz-2

We predict the structures of our designs using Boltz-2 ([Passaro et al., 2025](https://www.biorxiv.org/content/10.1101/2025.06.14.659707v1)).

Typically, to validate *in silico* binder designs, we predict the target-binder complex
structure and check for consistency with the original designed structure and binding
interface. Because the structure of the  target is known, but the binder structure is
not, we usually run structure prediction in single sequence mode for both the binder and
target sequences using only the target structure as a template. _Single sequence mode_
means that no multiple sequence alignment is used for the binder. This is important to
efficiently screen large numbers of binder designs where homology search is a bottleneck.

However, because we do support templates yet (coming soon!), in this tutorial we'll
use an MSA for the target instead of a template; this generally achieves the same
objective, allowing the model to accurately recapitulate the target's known structure.

Below, we compute an MSA for the target, and use it to predict the structure of all
designed sequences; this generally takes about 50 minutes.

In [43]:
# 1. Compute MSA for target to use for folding
target_msa = session.align.create_msa(target.sequence)

# 2. Create the complexes to fold
complexes_to_fold = []
for i, generated_structure in enumerate(generated_structures):  # for each structure
    for j in range(N_SEQS_PER_STRUCTURE):
        designed_binder_sequence = df.loc[(i, j)]["sequence"]
        complex = Complex(
            {"A": Protein(target.sequence), "B": Protein(designed_binder_sequence)}
        )
        # set MSAs to use for structure prediction
        #   note that `binder.msa = Protein.single_sequence_mode` needs to be set
        #   explicitly when running structure prediction without MSAs
        complex.get_protein(chain_id="A").msa = target_msa
        complex.get_protein(chain_id="B").msa = Protein.single_sequence_mode
        complexes_to_fold.append(complex)

# 3. Create the job
fold_job = session.fold.boltz_2.fold(complexes_to_fold)
print(fold_job)

# 4. Wait for the job to finish
_ = fold_job.wait_until_done(verbose=True)

/Users/tbepler/miniconda3/envs/openprotein/lib/python3.12/site-packages/openprotein/base.py:119: UserWarning: The requested payload is >1MB. There might be some delays or issues in processing. If the request fails, please try again with smaller sizes.
  warnings.warn(


num_records=1000 job_id='f47dbdc4-12a3-4639-8922-40302cc2aa4c' job_type=<JobType.embeddings_fold: '/embeddings/fold'> status=<JobStatus.PENDING: 'PENDING'> created_date=datetime.datetime(2026, 1, 21, 9, 59, 45, 820885, tzinfo=TzInfo(UTC)) start_date=None end_date=None prerequisite_job_id=None progress_message=None progress_counter=0 sequence_length=None


Waiting: 100%|██████████| 100/100 [48:09<00:00, 28.90s/it, status=SUCCESS] 


We retrieve the predicted structures as a list of `Complex` objects:

In [44]:
predicted_structures: list[Complex] = []
for structure in fold_job.get(verbose=True):
    # each prediction is a Structure object
    assert isinstance(structure, Structure)
    # since we only make one prediction per design, we extract the `Complex` of that one
    # prediction only, and append that to our list of predicted structures
    predicted_structures.append(structure[0])

Retrieving: 100%|██████████| 1000/1000 [00:57<00:00, 17.30it/s]


Let's visualize the first predicted structure to check that it looks reasonable:

In [45]:
visualize_cif(
    predicted_structures[0]
    .copy()
    .transform(
        # apply a rotation to make the binding site clearer
        # (its not necessary to understand how to define the rotation)
        R=(
            Rotation.from_euler("y", 180, degrees=True)
            * Rotation.from_euler("x", 90, degrees=True)
        ).as_matrix()
    )
    .to_string(),
    colors=[
        ColorSpec(chain_id="A", color="#b5e2f5"),  # target in blue
        ColorSpec(chain_id="B", color="#f4c30b"),  # binder in orange
        #ColorSpec(
        #    chain_id="A",
        #    color="#6bb50a",  # epitope in green
        #    positions=binding_sites,
        #    rep_type="ball_and_stick",
        #),
    ],
)

<IPython.core.display.Javascript object>

As desired, the predicted structure contains the target, and the binder close to the target chain.

In addition to the predicted structures, we also retrieve the predicted aligned errors
(PAEs), which we will use for computing metrics below. The [PAE](https://www.ebi.ac.uk/training/online/courses/alphafold/inputs-and-outputs/evaluating-alphafolds-predicted-structures-using-confidence-scores/pae-a-measure-of-global-confidence-in-alphafold-predictions/) is a structure prediction
confidence metric that has been [highly effective at identifying successful binders](https://www.nature.com/articles/s41467-023-38328-5).

In [46]:
predicted_paes: list[npt.NDArray[np.floating]] = fold_job.get_pae()

## Filter and select designs by metrics

We compute, filter, and select designs using standard structure prediction metrics and
thresholds adapted from the RFdiffusion ([Watson et al., 2023](https://www.nature.com/articles/s41586-023-06415-8)) and BoltzGen studies.

| Metric | Description | Ideal Value |
| --- | --- | --- |
| **RMSD** | Measures how closely the predicted structure of the *entire complex* matches the generated structure. | < 3.0 Å |
| **iPAE** | Confidence that the binder forms an interface with the target. | < 10 |
| **Binder RMSD** | Measures how closely the predicted structure of *just the binder* matches the generated structure. | < 2.0 Å |
| **Binder pLDDT** | Confidence in the predicted structure of the binder. | > 80 |

### Compute Metrics

Below, we compute these metrics and collate the metrics and designed sequences into a
dataframe for further analysis.

In [47]:
records = []  # collect metrics and designed sequences into a list of records
for i, generated_structure in enumerate(tqdm(generated_structures, mininterval=1.0)):
    for j in range(N_SEQS_PER_STRUCTURE):
        predicted_structure = predicted_structures[i * N_SEQS_PER_STRUCTURE + j]
        # compute overall rmsd
        rmsd = predicted_structure.rmsd(generated_structure)
        # compute ipae
        pae = predicted_paes[i * N_SEQS_PER_STRUCTURE + j].squeeze(0)
        ipae0 = np.mean(pae[: len(target), len(target) :])
        ipae1 = np.mean(pae[len(target) :, : len(target)])
        ipae = (ipae0 + ipae1) / 2
        # compute binder metrics
        generated_binder = generated_structure.get_protein(chain_id="B")
        predicted_binder = predicted_structure.get_protein(chain_id="B")
        binder_rmsd = predicted_binder.rmsd(generated_binder)
        binder_plddt = predicted_binder.plddt.mean()
        # get dataframe row containing designed sequence
        row = df.loc[(i, j)]
        # record all relevant data
        records.append(
            {
                "design_idx": row["design_idx"],
                "structure_idx": i,
                "sequence_idx": j,
                "rmsd": rmsd,
                "ipae": ipae,
                "binder_rmsd": binder_rmsd,
                "binder_plddt": binder_plddt,
                "score": row["score"],
                "sequence": row["sequence"],
            }
        )
df = pd.DataFrame.from_records(records).set_index(["structure_idx", "sequence_idx"])
df.head()

100%|██████████| 100/100 [00:00<00:00, 120.98it/s]


design_idx       rmsd       ipae  binder_rmsd  \
structure_idx sequence_idx                                                  
0             0                      0   6.356473   6.783213     1.603597   
              1                      1   6.383963   7.107962     0.845479   
              2                      2  12.240748  13.763231     0.930982   
              3                      3   5.423607   5.302590     0.855966   
              4                      4  14.176849   9.529541     0.942314   

                            binder_plddt       score  \
structure_idx sequence_idx                             
0             0                89.782974 -113.704020   
              1                89.049973 -115.618840   
              2                90.488663 -112.545610   
              3                83.076797 -120.459770   
              4                90.865143 -117.207985   

                                                                     sequence  
structure_idx sequence_idx                                                     
0             0             EVQLVESGGGLVQPGGSLRLSCAASGNFDFFAKLYGWFRQAPGKGR...  
              1             EVQLVESGGGLVQPGGSLRLSCAASGNFDFFAKYYGWFRQAPGKGR...  
              2             EVQLVESGGGLVQPGGSLRLSCAASGNYDFFSKLYGWFRQAPGKGR...  
              3             EVQLVESGGGLVQPGGSLRLSCAASGNFDFFAKLAGWFRQAPGKGR...  
              4             EVQLVESGGGLVQPGGSLRLSCAASGNFDFFARYYGWFRQAPGKGR...

### Filter and select designs by metrics

We start by filtering the designs based on the ideal metric thresholds.

In [50]:
df_filtered = df[
    (df["rmsd"] < 3.0)
    & (df["ipae"] < 10)
    & (df["binder_rmsd"] < 2.0)
    & (df["binder_plddt"] > 80)
]
print("# designs passing filters", len(df_filtered))
print(
    "# unique structures passing filters",
    df_filtered.index.get_level_values("structure_idx").nunique(),
)

# designs passing filters 59
# unique structures passing filters 36


Looks like we have a good number of designs meeting the ideal metric thresholds!

Next, we rank the designs based on iPAE to prioritize designs with high confidence of
interaction. We'll also select just the top design per unique structure, to select for
a diverse set of binders.

In [124]:
df_selected = (
    # rank by ipae
    df_filtered.reset_index().sort_values(by="ipae")
    # select best sequence per structure
    .groupby("structure_idx", sort=False).first()
    # set dataframe index
    .reset_index().set_index(["structure_idx", "sequence_idx"])
)
df_selected.head(7)

,,design_idx,rmsd,ipae,binder_rmsd,binder_plddt,score,sequence
structure_idx,sequence_idx,,,,,,,
70,3,703,2.524818,4.327188,1.641544,86.656006,-98.528120,EVQLVESGGGLVQPGGSLRLSCAASGRFDFEKLLFGWFRQAPGKGR...
90,1,901,2.144689,4.559683,1.355856,87.623535,-111.570320,EVQLVESGGGLVQPGGSLRLSCAASGNYYFFAKYLGWFRQAPGKGR...
69,7,697,2.381741,4.586547,0.912900,94.181946,-104.432850,EVQLVESGGGLVQPGGSLRLSCAASGRFDIGKLAYGWFRQAPGKGR...
43,7,437,2.802682,4.699512,0.749971,91.440849,-93.867730,EVQLVESGGGLVQPGGSLRLSCAASGRFDFTALLLGWFRQAPGKGR...
56,3,563,2.523395,4.736653,0.908991,91.884071,-100.993416,EVQLVESGGGLVQPGGSLRLSCAASGRFVWWQLAVGWFRQAPGKGR...
27,1,271,2.994201,4.746180,0.871660,89.266937,-90.719250,EVQLVESGGGLVQPGGSLRLSCAASGRFDFQKLAYGWFRQAPGKGR...
7,3,73,2.377406,4.766557,0.943879,94.753258,-88.238200,EVQLVESGGGLVQPGGSLRLSCAASGNFDFQQLAYGWFRQAPGKGR...


We now have a ranked list of promising binder designs!

Before we send them off for experimental validation, we should visually inspect their
stuctures for any anomalies. For example, below we visualize the predicted structure of
the top ranked design superimposed onto the corresponding generated structure (light
gray) and see that it looks reasonable on visual inspection:

In [125]:
# get predicted and generated structure of top design
design_idx, structure_idx = df_selected.reset_index().iloc[0][['design_idx', 'structure_idx']]
predicted_structure = predicted_structures[design_idx]
generated_structure = generated_structures[structure_idx]
# superimpose predicted structure on the generate structure
predicted_structure = predicted_structure.copy().superimpose_onto(generated_structure)
# visualize
visualize_cif(
    Complex(
        {
            "A_predicted": predicted_structure.get_protein(chain_id="A"),
            "B_predicted": predicted_structure.get_protein(chain_id="B"),
            "A_generated": generated_structure.get_protein(chain_id="A"),
            "B_generated": generated_structure.get_protein(chain_id="B"),
        }
    ).to_string(),
    colors=[
        ColorSpec(chain_id="A_predicted", color="#b5e2f5"),
        ColorSpec(chain_id="B_predicted", color="#f4c30b"),
        ColorSpec(chain_id="A_generated", color="#F2F0EF"),
        ColorSpec(chain_id="B_generated", color="#F2F0EF"),
    ],
)

<IPython.core.display.Javascript object>

In [126]:
# get predicted and generated structure of the second best
design_idx, structure_idx = df_selected.reset_index().iloc[1][['design_idx', 'structure_idx']]
predicted_structure = predicted_structures[design_idx]
generated_structure = generated_structures[structure_idx]
# superimpose predicted structure on the generate structure
predicted_structure = predicted_structure.copy().superimpose_onto(generated_structure)
# visualize
visualize_cif(
    Complex(
        {
            "A_predicted": predicted_structure.get_protein(chain_id="A"),
            "B_predicted": predicted_structure.get_protein(chain_id="B"),
            "A_generated": generated_structure.get_protein(chain_id="A"),
            "B_generated": generated_structure.get_protein(chain_id="B"),
        }
    ).to_string(),
    colors=[
        ColorSpec(chain_id="A_predicted", color="#b5e2f5"),
        ColorSpec(chain_id="B_predicted", color="#f4c30b"),
        ColorSpec(chain_id="A_generated", color="#F2F0EF"),
        ColorSpec(chain_id="B_generated", color="#F2F0EF"),
    ],
)

<IPython.core.display.Javascript object>

After visually confirming the designs and further filtering based on any additional
metrics you may have in mind (e.g. metrics relevant to your specific assay), the designs
can then be sent off for experimental testing!

# Conclusion

In this walkthrough, we've demonstrated how to design nanobody binders for a target of
interest using BoltzGen and PoET-2. We validated the designs using in-silico metrics and visualized them to ensure
their viability. The top-ranked designs from this workflow can be:

1. Expressed and purified for experimental validation
2. Tested for binding affinity
3. Further optimized through additional rounds of design, for example, with
   [OpenProtein.AI's property regression models](https://docs.openprotein.ai/python-api/property-regression-models/index.html).

### Other resources

Read more about our binder design workflows and other de novo design tools here:
- [Designing miniprotein binders with RFdiffusion](https://docs.openprotein.ai/walkthroughs/Protein_protein_binder_design_with_RFdiffusion.html)
- [Inverse folding for protein redesign](https://docs.openprotein.ai/walkthroughs/PoET-2_inverse_folding.html)
- [Antibody lead optimization](https://docs.openprotein.ai/walkthroughs/antibody-engineering.html)

or see the detailed API references
- [BoltzGen](https://docs.openprotein.ai/python-api/api-reference/models.html#boltzgen)
- [proteinMPNN](https://docs.openprotein.ai/python-api/api-reference/models.html#proteinmpnn)
- [Boltz-2](https://docs.openprotein.ai/python-api/api-reference/fold.html#openprotein.fold.Boltz2Model)
- [PoET-2](https://docs.openprotein.ai/python-api/api-reference/embedding.html#openprotein.embeddings.PoET2Model)